In [21]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [22]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, MultiProductContextEmbeddings

In [23]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [24]:
# ── Datos ──────────────────────────────────────────────────────────
N_UPCS        = 5
TRAIN_FRAC    = 0.8
SMOOTH_WINDOW = 8

BETA_EDA = -2

# ── Entrenamiento (reducido para que el search sea rápido) ─────────
N_EPOCHS_P0 = 350
N_EPOCHS_P1 = 350
N_EPOCHS_P2 = 400
PATIENCE    = 30
ES_PATIENCE = 70

# ── Checkpoints ────────────────────────────────────────────────────
CKPT_DIR = Path("../results/checkpoints/hparam")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
loader = DominickDataLoader()
df     = loader.load("elasticity_dataset.csv")
print(f"Dataset shape: {df.shape}")

encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id",    sort=True)
n_stores = len(store_cats)
n_weeks  = len(week_cats)
print(f"Tiendas: {n_stores}  |  Semanas: {n_weeks}")

sorted_weeks   = sorted(df["week_id"].unique())
week_threshold = sorted_weeks[int(len(sorted_weeks) * TRAIN_FRAC)]
train_df_raw   = df[df["week_id"] < week_threshold].copy()

mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)
full_wide  = mp_builder.transform(df)
train_wide = full_wide[full_wide["week_id"] < week_threshold].copy()
val_wide   = full_wide[full_wide["week_id"] >= week_threshold].copy()
n_upcs     = mp_builder.n

store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}
for w in [train_wide, val_wide]:
    w["store_code"] = w["store_code"].map(store_map)
    w["week_id"]    = w["week_id"].map(week_map)

print(f"Train: {len(train_wide):,}  |  Val: {len(val_wide):,}")
print(f"UPCs seleccionados: {n_upcs}")

Dataset shape: (463722, 30)
Tiendas: 70  |  Semanas: 302
Train: 15,822  |  Val: 3,986
UPCs seleccionados: 5


In [26]:
train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

for i in range(n_upcs):
    col = f"log_liters_{i}"
    for df_w in [train_wide_s, val_wide_s]:
        df_w[col] = (
            df_w.groupby("store_code")[col]
            .transform(lambda s: s.rolling(window=SMOOTH_WINDOW, min_periods=1).mean())
        )

train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
val_ds      = MultiProductDataset(val_wide,     n=n_upcs)

print("Datasets listos")

Datasets listos


In [27]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, phase_name="", verbose=False):

    best_val_loss = float("inf")
    no_improve    = 0
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        total_loss, total_denom = 0.0, 0.0

        for batch in train_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, eps_hat, aux = model(batch, return_parts=True)
                    loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                        aux["w"], aux["ddBx"], aux["u"],
                                        aux["Bx"], aux["IBx"],
                                        model.head.param_head._pairs)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, eps_hat, aux = model(batch, return_parts=True)
                loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                    aux["w"], aux["ddBx"], aux["u"],
                                    aux["Bx"], aux["IBx"],
                                    model.head.param_head._pairs)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom        = obs_mask.sum().item()
            total_loss  += logs["loss"].item() * denom
            total_denom += denom

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        val_loss_sum, val_denom = 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device) for k, v in batch.items()}
                y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
                obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

                y_hat, eps_hat, aux = model(batch, return_parts=True)
                _, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                  aux["w"], aux["ddBx"], aux["u"],
                                  aux["Bx"], aux["IBx"],
                                  model.head.param_head._pairs)
                denom        = obs_mask.sum().item()
                val_loss_sum += logs["loss"].item() * denom
                val_denom    += denom

        val_loss = val_loss_sum / max(val_denom, 1.0)
        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr < prev_lr:
            no_improve = 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping en época {epoch+1}")
            break

    return best_val_loss

print("run_training definida")

run_training definida


In [28]:
HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64_32":  (128, 64, 32),
    "64_32_16":   (64, 32, 16),
    "128_64":     (128, 64),
}

def compute_global_r2(model, val_loader, device):
    """R² global en log-liters sobre todas las observaciones de validación."""
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)
            y_hat, _, _ = model(batch, return_parts=True)

            mask = obs_mask.bool()
            all_true.append(y_true[mask].cpu())
            all_pred.append(y_hat[mask].cpu())

    y_true_all = torch.cat(all_true)
    y_pred_all = torch.cat(all_pred)

    ss_res = ((y_true_all - y_pred_all) ** 2).sum()
    ss_tot = ((y_true_all - y_true_all.mean()) ** 2).sum()
    return float(1.0 - ss_res / ss_tot)


def compute_elasticity_score(model, val_loader, device,
                              elast_min=-5.0, elast_max=0.0):
    """
    Score de coherencia de elasticidades propias (eps_hat) en validación.
    Combina:
      - fracción de elasticidades dentro del rango económico [elast_min, elast_max]
      - penalización si la mediana se aleja mucho del prior BETA_EDA
    Devuelve un valor entre 0 y 1 (mayor = más coherente).
    """
    model.eval()
    all_elast = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            obs_mask = torch.stack([batch[f"obs_mask_{i}"] for i in range(model.n)], dim=1).bool()
            _, eps_hat, _ = model(batch, return_parts=True)
            all_elast.append(eps_hat[obs_mask].cpu())
    elast = torch.cat(all_elast).numpy()

    # Fracción en rango económico razonable
    in_range = float(((elast >= elast_min) & (elast <= elast_max)).mean())

    # Penalización por desviación de la mediana respecto al prior EDA
    median_e = float(np.median(elast))
    deviation = max(0.0, abs(median_e - BETA_EDA) - 0.3)   # tolerancia ±0.3
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)     # normalizado a [0,1]

    score = in_range * (1.0 - prior_penalty)
    return float(score), float(median_e), float(in_range)


def build_and_train(params, trial_id=0):
    """
    Entrena las 3 fases con los hiperparámetros dados.
    Devuelve (r2_global, elast_score) para optimización multi-objetivo.
    """
    n_knots          = params["N_KNOTS"]
    hidden           = HIDDEN_OPTIONS[params["HIDDEN_KEY"]] 
    dropout          = params["DROPOUT"]
    d_store          = params.get("D_STORE", 16)
    act              = params.get("ACT", "gelu")
    lr_p0            = params["LR_P0"]
    lr_p1            = params["LR_P1"]
    lr_p2            = params["LR_P2"]
    lambda_smooth_p2 = params["LAMBDA_SMOOTH_P2"]
    lambda_pos_p2    = params["LAMBDA_POS_P2"]
    batch_size       = params["BATCH_SIZE"]

    ckpt_p0 = CKPT_DIR / f"trial{trial_id}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"trial{trial_id}_phase1.pt"
    ckpt_p2 = CKPT_DIR / f"trial{trial_id}_phase2.pt"

    # ── DataLoaders ───────────────────────────────────────────────
    loader_factory  = DataLoaderFactory(num_workers=0, pin_memory=True)
    train_loader_p0 = loader_factory.create_train_loader(
        train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader_p0   = loader_factory.create_eval_loader(
        val_ds_p0, batch_size=batch_size, shuffle=False
    )
    train_loader    = loader_factory.create_train_loader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader      = loader_factory.create_eval_loader(
        val_ds, batch_size=batch_size, shuffle=False
    )

    # ── Splines ────────────────────────────────────────────────────
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i    = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=n_knots, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    # ── Context builder ─────────────────────────────────────────────
    cb = MultiProductContextEmbeddings(
        n=n_upcs,
        n_stores=n_stores,
        d_store=d_store,
    )

    def make_model(enforce_negative_beta, use_cross):
        head = IntegrableDemandHead(
            context_dim=cb.out_dim,
            K_splines=n_knots,
            n=n_upcs,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        return ICDN(
            context_builder=cb,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)

    # ── FASE 0 ─────────────────────────────────────────────────────
    m0 = make_model(enforce_negative_beta=True, use_cross=False)
    with torch.no_grad():
        m0.head.param_head.head_w.weight.zero_()
        m0.head.param_head.head_w.bias.zero_()
    m0.head.param_head.head_w.weight.requires_grad_(False)
    m0.head.param_head.head_w.bias.requires_grad_(False)

    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-BETA_EDA, dtype=torch.float32)) - 1.0
    )
    with torch.no_grad():
        m0.head.param_head.head_beta.weight.zero_()
        m0.head.param_head.head_beta.bias.fill_(beta_raw_init)

    loss_p0 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p0 = torch.optim.AdamW(
        [
            {"params": decay,    "weight_decay": 1e-5},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=lr_p0,
    )
    sch_p0  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE, ckpt_p0, device, "P0")

    # ── FASE 1 ─────────────────────────────────────────────────────
    m1 = make_model(enforce_negative_beta=True, use_cross=False)
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    m1.head.param_head.head_w.weight.requires_grad_(True)
    m1.head.param_head.head_w.bias.requires_grad_(True)

    loss_p1 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p1 = torch.optim.AdamW(
        [
            {"params": decay,    "weight_decay": 1e-5},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=lr_p1,
    )
    
    sch_p1  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE, ckpt_p1, device, "P1")

    # ── FASE 2 ─────────────────────────────────────────────────────
    m2 = make_model(enforce_negative_beta=True, use_cross=True)
    state = torch.load(ckpt_p1, map_location=device)
    state.pop("head.param_head._pairs", None)
    m2.load_state_dict(state, strict=False)

    m2.head.param_head.head_w.weight.requires_grad_(True)
    m2.head.param_head.head_w.bias.requires_grad_(True)
    with torch.no_grad():
        m2.head.param_head.head_cross.weight.zero_()
        m2.head.param_head.head_cross.bias.zero_()

    loss_p2 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth_p2,
        lambda_pos=lambda_pos_p2,
        reduction="none",
    )
    decay, no_decay = [], []
    for name, p in m2.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p2 = torch.optim.AdamW(
        [
            {"params": decay,    "weight_decay": 1e-5},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=lr_p2,
    )
    sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p2, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m2, train_loader, val_loader, loss_p2,
                 opt_p2, sch_p2, N_EPOCHS_P2, ES_PATIENCE, ckpt_p2, device, "P2")

    # ── Métricas finales sobre el mejor checkpoint de Fase 2 ───────
    m2.load_state_dict(torch.load(ckpt_p2, map_location=device))

    r2                        = compute_global_r2(m2, val_loader, device)
    elast_score, median_e, in_range = compute_elasticity_score(m2, val_loader, device)

    print(f"  R²={r2:.4f}  |  elast_score={elast_score:.4f}"
          f"  (mediana={median_e:.3f}, en_rango={in_range:.2%})")

    # Limpiar checkpoints intermedios
    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)

    return r2, elast_score

print("Métricas y build_and_train definidas")

Métricas y build_and_train definidas


In [29]:
def objective(trial):
    params = {
        "N_KNOTS":          trial.suggest_int("N_KNOTS", 2, 16),
        "HIDDEN_KEY":       trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
        "DROPOUT":          trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":            trial.suggest_float("LR_P0",  1e-4, 1e-2, log=True),
        "LR_P1":            trial.suggest_float("LR_P1",  1e-5, 5e-3, log=True),
        "LR_P2":            trial.suggest_float("LR_P2",  1e-5, 1e-3, log=True),
        "LAMBDA_SMOOTH_P2": trial.suggest_float("LAMBDA_SMOOTH_P2", 1e-5, 0.2, log=True),
        "LAMBDA_POS_P2":    trial.suggest_float("LAMBDA_POS_P2",    0.05, 0.5),
        "BATCH_SIZE":       trial.suggest_categorical("BATCH_SIZE", [16, 32, 64]),
    }

    print(f"\n{'='*60}")
    print(f"Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print(f"{'='*60}")

    r2, elast_score = build_and_train(params, trial_id=trial.number)
    return r2, elast_score

In [30]:
study = optuna.create_study(
    directions=["maximize", "maximize"],   # maximizar R² Y elast_score
    study_name="hparam_pareto",
    storage="sqlite:///../results/hparam_pareto.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=2)

print(f"\nTrials completados: {len(study.trials)}")
print(f"Trials Pareto-óptimos: {len(study.best_trials)}")

[I 2026-03-10 10:37:51,460] A new study created in RDB with name: hparam_pareto



Trial 0
  N_KNOTS: 3
  HIDDEN_KEY: 64_32
  DROPOUT: 0.2203626634059746
  LR_P0: 0.0010068131284112877
  LR_P1: 5.596027700586267e-05
  LR_P2: 0.0005715169083582696
  LAMBDA_SMOOTH_P2: 0.0006256975251284771
  LAMBDA_POS_P2: 0.3765945102887505
  BATCH_SIZE: 64


[I 2026-03-10 11:35:25,496] Trial 0 finished with values: [0.494195818901062, 0.9062623079952737] and parameters: {'N_KNOTS': 3, 'HIDDEN_KEY': '64_32', 'DROPOUT': 0.2203626634059746, 'LR_P0': 0.0010068131284112877, 'LR_P1': 5.596027700586267e-05, 'LR_P2': 0.0005715169083582696, 'LAMBDA_SMOOTH_P2': 0.0006256975251284771, 'LAMBDA_POS_P2': 0.3765945102887505, 'BATCH_SIZE': 64}.


  R²=0.4942  |  elast_score=0.9063  (mediana=-2.043, en_rango=90.63%)

Trial 1
  N_KNOTS: 9
  HIDDEN_KEY: 64_32_16
  DROPOUT: 0.2122592921813092
  LR_P0: 0.0004960588946681797
  LR_P1: 0.003304616930502365
  LR_P2: 0.0005093535961145223
  LAMBDA_SMOOTH_P2: 0.003759692455526979
  LAMBDA_POS_P2: 0.15396231526968818
  BATCH_SIZE: 32


[I 2026-03-10 12:45:14,179] Trial 1 finished with values: [0.3811754584312439, 0.8576604962583695] and parameters: {'N_KNOTS': 9, 'HIDDEN_KEY': '64_32_16', 'DROPOUT': 0.2122592921813092, 'LR_P0': 0.0004960588946681797, 'LR_P1': 0.003304616930502365, 'LR_P2': 0.0005093535961145223, 'LAMBDA_SMOOTH_P2': 0.003759692455526979, 'LAMBDA_POS_P2': 0.15396231526968818, 'BATCH_SIZE': 32}.


  R²=0.3812  |  elast_score=0.8577  (mediana=-1.817, en_rango=85.77%)

Trials completados: 2
Trials Pareto-óptimos: 1


In [31]:
print(f"\n{'='*70}")
print(f"{'Trial':>6}  {'R²':>8}  {'ElastScore':>11}  Parámetros")
print(f"{'='*70}")

pareto = sorted(study.best_trials, key=lambda t: t.values[0], reverse=True)
for t in pareto:
    r2_val = t.values[0]
    es_val = t.values[1]
    params_str = ", ".join(f"{k.replace('params_','')}={v}"
                           for k, v in t.params.items())
    print(f"  {t.number:4d}  {r2_val:8.4f}  {es_val:11.4f}  {params_str}")


 Trial        R²   ElastScore  Parámetros
     0    0.4942       0.9063  N_KNOTS=3, HIDDEN_KEY=64_32, DROPOUT=0.2203626634059746, LR_P0=0.0010068131284112877, LR_P1=5.596027700586267e-05, LR_P2=0.0005715169083582696, LAMBDA_SMOOTH_P2=0.0006256975251284771, LAMBDA_POS_P2=0.3765945102887505, BATCH_SIZE=64


In [32]:
df_trials = study.trials_dataframe()
param_cols = [c for c in df_trials.columns if c.startswith("params_")]
val_cols   = [c for c in df_trials.columns if c.startswith("values_")]

df_show = (df_trials[["number"] + val_cols + param_cols]
           .rename(columns={"values_0": "R2", "values_1": "elast_score"})
           .dropna(subset=["R2"])
           .sort_values("R2", ascending=False))

print(df_show.head(15).to_string(index=False))

 number       R2  elast_score  params_BATCH_SIZE  params_DROPOUT params_HIDDEN_KEY  params_LAMBDA_POS_P2  params_LAMBDA_SMOOTH_P2  params_LR_P0  params_LR_P1  params_LR_P2  params_N_KNOTS
      0 0.494196     0.906262                 64        0.220363             64_32              0.376595                 0.000626      0.001007      0.000056      0.000572               3
      1 0.381175     0.857660                 32        0.212259          64_32_16              0.153962                 0.003760      0.000496      0.003305      0.000509               9
